# Notebook 05: LLM-Based Fabrication Validation of v1 Summaries

Validates AI-generated clinical summaries (v1 prompt) against the ground truth
fabrication dataset using an LLM as judge, following the approach in the
[OpenAI cookbook: Using reasoning for data validation](https://github.com/openai/openai-cookbook/blob/main/examples/o1/Using_reasoning_for_data_validation.ipynb).

## Workflow
```
v1_summary_paths.csv  +  ai_fabrications_dataset.xlsx  (14 confirmed fabrications)
  ↓ extract text from .docx summaries (python-docx)
  ↓ Stage 1 — blind detection
  |   send (summary_text, feature_name) → LLM
  |   → {is_fabrication: bool, confidence: float, issue: str}
  ↓ Stage 2 — issue comparison
  |   send (model_issue, ground_truth_ai_fab_comment) → judge LLM
  |   → {issues_match: bool}
  ↓ precision / recall / F1 vs ground truth labels
  ↓ results table + figures → reports/
```

**Ground truth:** all 14 cases are confirmed fabrications (`gt_is_fabrication=True`)  
**Model:** `gpt-4o` (default) — swap `VALIDATION_MODEL` to `o1-mini` or `o1` for deeper reasoning

## 0. Setup & Imports

In [ ]:
import os
import sys
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import precision_score, recall_score, f1_score
from docx import Document
from tqdm.notebook import tqdm

load_dotenv()

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT",
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
    r"\Documents\GitHub\llm_summarization_br_ca"))
DATA_PRIVATE = Path(os.getenv("DATA_PRIVATE_DIR",
    r"C:\Users\jamesr4\loc\data_private"))

FAB_XLSX      = DATA_PRIVATE / "raw" / "ai_fabrications_dataset.xlsx"
SUMMARY_PATHS = DATA_PRIVATE / "raw" / "v1_summary_paths.csv"
RUN_OUT_DIR   = PROJECT_ROOT / "experiments" / "runs" / "llm_validation"
REPORTS_DIR   = PROJECT_ROOT / "reports"

RUN_OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Model config ──────────────────────────────────────────────────────────────
# Swap to "o1-mini" or "o1" for stronger reasoning
VALIDATION_MODEL  = "gpt-4o"
ISSUE_JUDGE_MODEL = "gpt-4o"

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
print(f"OpenAI key loaded : {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"Validation model  : {VALIDATION_MODEL}")
print(f"Issue judge model : {ISSUE_JUDGE_MODEL}")

## 1. Load Fabrication Cases & Summary Paths

In [ ]:
df_fab   = pd.read_excel(FAB_XLSX)
df_paths = pd.read_csv(SUMMARY_PATHS)

ai_cols = [c for c in df_fab.columns if c.endswith("_status_ai")]
for c in ai_cols:
    df_fab[c] = pd.to_numeric(df_fab[c], errors="coerce")

df_fab["surgeon_last"] = df_fab["surgeon"].str.split(",").str[0].str.strip()

df = df_fab.merge(df_paths, on="mrn", how="left")

# Fabricated features per patient
df["fab_features"] = df.apply(
    lambda row: [c.replace("_status_ai", "") for c in ai_cols if row[c] == 3],
    axis=1,
)
# All 14 are confirmed fabrications
df["gt_is_fabrication"] = True

print(f"Cases loaded: {len(df)} | Summary paths found: {df['summary_path'].notna().sum()}")
print()
for _, row in df.iterrows():
    print(f"  MRN {int(row['mrn']):<12} {row['surgeon_last']:<12} {row['patient_initials']:<5}"
          f" | feats={row['fab_features']}")
    print(f"    file  : {Path(row['summary_path']).name}")
    print(f"    issue : {row.get('ai_fab_comment', '')}")

## 2. Extract Text from .docx Summaries

In [ ]:
def extract_docx_text(path: str | Path) -> str:
    """Extract paragraph + table text from a .docx file."""
    try:
        doc = Document(str(path))
        parts = []
        for para in doc.paragraphs:
            t = para.text.strip()
            if t:
                parts.append(t)
        for table in doc.tables:
            for trow in table.rows:
                cells = " | ".join(
                    c.text.strip() for c in trow.cells if c.text.strip()
                )
                if cells:
                    parts.append(cells)
        return "\n".join(parts).strip()
    except Exception as exc:
        return f"[EXTRACTION_ERROR: {exc}]"


df["summary_text"] = df["summary_path"].apply(
    lambda p: extract_docx_text(p) if pd.notna(p) else ""
)

# Report extraction results
for _, row in df.iterrows():
    n_chars = len(row["summary_text"])
    status  = "OK" if n_chars > 100 else "EMPTY/ERROR"
    print(f"  MRN {int(row['mrn']):<12} {Path(row['summary_path']).name:<40} "
          f"{n_chars:>6} chars  [{status}]")

## 3. Feature Descriptions

Human-readable descriptions used in the validation prompt.

In [ ]:
FEATURE_DESCRIPTIONS = {
    "lesion_size": (
        "The measured size of the breast lesion (tumor/mass) as reported in imaging "
        "(mammography, ultrasound, or MRI) or pathology."
    ),
    "laterality": "Which breast is affected (left or right).",
    "lesion_location": "The anatomical location of the lesion within the breast (e.g., quadrant, clock position).",
    "calcifications_asymmetry": "Presence and characterization of calcifications or breast asymmetry on imaging.",
    "additional_enhancement_mri": "Presence of additional enhancement lesions seen on MRI beyond the index lesion.",
    "extent": (
        "The extent or distribution of disease (e.g., focal vs. diffuse, span of "
        "calcifications, multicentric vs. multifocal)."
    ),
    "accurate_clip_placement": "Whether the biopsy clip is accurately placed at the lesion site.",
    "workup_recommendation": (
        "Additional workup recommended (e.g., MRI, staging scans, biopsy, "
        "genetic testing) based on the clinical picture."
    ),
    "Lymph node": "Lymph node status — presence or absence of suspicious axillary lymph nodes.",
    "chronology_preserved": (
        "Whether dates and chronological sequence of events (imaging, biopsy, "
        "pathology) are accurately preserved in the summary."
    ),
    "biopsy_method": "The biopsy technique used (e.g., ultrasound-guided core biopsy, stereotactic, surgical excision).",
    "invasive_component_size_pathology": (
        "The size of the invasive carcinoma component as measured on pathology (not imaging)."
    ),
    "histologic_diagnosis": "The pathologic diagnosis / histologic type (e.g., invasive ductal, invasive lobular, DCIS).",
    "receptor": (
        "Hormone receptor and HER2 status: ER, PR, HER2 (e.g., ER+/PR+/HER2−, "
        "triple-negative, triple-positive)."
    ),
}

print(f"Feature descriptions defined: {len(FEATURE_DESCRIPTIONS)}")

## 4. Stage 1 — Blind Fabrication Detection

Send `(summary_text, feature_name)` to the LLM. The model sees **only the summary** — no hint about the known issue. Returns JSON `{is_fabrication, confidence, issue}`.

In [ ]:
SYSTEM_PROMPT = """You are a clinical data quality reviewer specializing in breast \
oncology. You review AI-generated surgical planning summaries for factual accuracy.

Your task is to assess whether a specific clinical feature is accurately reported \
in an AI summary. Use your clinical knowledge to identify values that are \
implausible, internally inconsistent, or contradicted within the document.

Respond ONLY with a JSON object. Do not include any other text."""


def validate_fabrication(summary_text: str, feature_name: str, mrn: int) -> dict:
    """
    Stage 1: Ask the LLM to check if `feature_name` is accurately reported
    in the summary. Returns dict with keys:
      is_fabrication (bool), confidence (float 0-1), issue (str)
    """
    feat_desc = FEATURE_DESCRIPTIONS.get(
        feature_name, feature_name.replace("_", " ").title()
    )

    user_content = f"""Review the following AI-generated breast oncology summary and \
assess the accuracy of ONE specific clinical feature.

FEATURE TO CHECK: {feature_name.replace('_', ' ').upper()}
Feature description: {feat_desc}

AI SUMMARY:
---
{summary_text}
---

Determine whether the above feature as stated in the summary contains a fabrication \
(an incorrect, implausible, or internally inconsistent value). Use your clinical \
expertise to reason about plausibility.

Return a JSON object with exactly these keys:
{{
  "is_fabrication": true or false,
  "confidence": float between 0 and 1,
  "issue": "description of the fabrication found, or empty string if none"
}}"""

    # o1 models do not support system role — use user role only
    is_o1 = VALIDATION_MODEL.startswith("o1")
    messages = (
        [{"role": "user",
          "content": SYSTEM_PROMPT + "\n\n" + user_content}]
        if is_o1
        else [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ]
    )

    kwargs = {"model": VALIDATION_MODEL, "messages": messages}
    if not is_o1:
        kwargs["response_format"] = {"type": "json_object"}
        kwargs["temperature"] = 0

    resp    = client.chat.completions.create(**kwargs)
    content = resp.choices[0].message.content.strip()

    # Parse — strip markdown fences if present
    if content.startswith("```"):
        content = "\n".join(content.split("\n")[1:-1])
    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        result = {"is_fabrication": None, "confidence": None,
                  "issue": content, "parse_error": True}

    result["mrn"]          = mrn
    result["feature_name"] = feature_name
    result["raw_response"] = content
    return result


print("validate_fabrication() defined")

In [ ]:
RESULTS_CACHE = RUN_OUT_DIR / "stage1_results.json"

if RESULTS_CACHE.exists():
    with open(RESULTS_CACHE) as f:
        stage1_results: dict = json.load(f)
    print(f"Loaded {len(stage1_results)} cached Stage 1 results")
else:
    stage1_results = {}

# Build task list — one entry per (mrn, feature)
tasks = []
for _, row in df.iterrows():
    mrn = int(row["mrn"])
    for feat in row["fab_features"]:
        key = f"{mrn}_{feat}"
        if key not in stage1_results:
            tasks.append({
                "key":          key,
                "mrn":          mrn,
                "feature":      feat,
                "summary_text": row["summary_text"],
            })

print(f"Tasks to run: {len(tasks)} (skipping {len(stage1_results)} cached)")

def _run_task(task):
    return task["key"], validate_fabrication(
        task["summary_text"], task["feature"], task["mrn"]
    )

failed = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(_run_task, t): t for t in tasks}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Stage 1"):
        try:
            key, result = fut.result()
            stage1_results[key] = result
            with open(RESULTS_CACHE, "w") as f:
                json.dump(stage1_results, f, indent=2, default=str)
        except Exception as exc:
            t = futures[fut]
            print(f"  ERROR {t['key']}: {exc}")
            failed.append(t["key"])

print(f"\nStage 1 complete — {len(stage1_results)} results, {len(failed)} failed")

## 5. Stage 2 — Issue Comparison

For cases where the LLM predicted `is_fabrication=True`, compare its `issue` description against the known `ai_fab_comment` using a judge LLM. Returns `True` if they address the same clinical problem.

In [ ]:
def validate_issue(model_issue: str, ground_truth_issue: str) -> bool:
    """
    Stage 2: Ask the judge LLM whether model_issue and ground_truth_issue
    address the same underlying clinical problem. Returns True/False.
    Based on OpenAI cookbook validate_issue() pattern.
    """
    prompt = f"""You are a medical expert assistant reviewing the quality of an \
LLM-generated clinical answer.

The model was asked to identify a fabrication in an AI-generated breast oncology \
summary. Your task:
  - Compare the model-generated issue description with the correct known issue.
  - Determine if they address the same underlying clinical inaccuracy, even if \
phrased differently.
  - Focus on the clinical concept and implication, not exact wording.

If they describe the same issue, respond: True
If they describe different issues, respond: False
Respond with ONLY a single word: True or False.

Model-generated issue : {model_issue}
Known correct issue   : {ground_truth_issue}"""

    is_o1 = ISSUE_JUDGE_MODEL.startswith("o1")
    messages = [{"role": "user", "content": prompt}]
    kwargs = {"model": ISSUE_JUDGE_MODEL, "messages": messages}
    if not is_o1:
        kwargs["temperature"] = 0

    resp = client.chat.completions.create(**kwargs)
    answer = resp.choices[0].message.content.strip()
    return answer.lower().startswith("true")


print("validate_issue() defined")

In [ ]:
STAGE2_CACHE = RUN_OUT_DIR / "stage2_results.json"

if STAGE2_CACHE.exists():
    with open(STAGE2_CACHE) as f:
        stage2_results: dict = json.load(f)
    print(f"Loaded {len(stage2_results)} cached Stage 2 results")
else:
    stage2_results = {}

# Build issue comparison tasks for cases where both LLM and GT say is_fabrication=True
issue_tasks = []
for _, row in df.iterrows():
    mrn = int(row["mrn"])
    known_issue = str(row.get("ai_fab_comment", "") or "")
    for feat in row["fab_features"]:
        key = f"{mrn}_{feat}"
        s1  = stage1_results.get(key, {})
        if s1.get("is_fabrication") is True and key not in stage2_results:
            issue_tasks.append({
                "key":         key,
                "model_issue": str(s1.get("issue", "")),
                "gt_issue":    known_issue,
            })

print(f"Issue comparison tasks: {len(issue_tasks)} (skipping {len(stage2_results)} cached)")

def _run_issue(task):
    match = validate_issue(task["model_issue"], task["gt_issue"])
    return task["key"], match

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(_run_issue, t): t for t in issue_tasks}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Stage 2"):
        try:
            key, match = fut.result()
            stage2_results[key] = match
            with open(STAGE2_CACHE, "w") as f:
                json.dump(stage2_results, f, indent=2)
        except Exception as exc:
            t = futures[fut]
            print(f"  ERROR {t['key']}: {exc}")

print(f"\nStage 2 complete — {len(stage2_results)} results")

## 6. Build Results Table

In [ ]:
rows = []
for _, row in df.iterrows():
    mrn        = int(row["mrn"])
    known_issue = str(row.get("ai_fab_comment", "") or "")
    for feat in row["fab_features"]:
        key = f"{mrn}_{feat}"
        s1  = stage1_results.get(key, {})

        pred_fab   = s1.get("is_fabrication")
        confidence = s1.get("confidence")
        model_issue = str(s1.get("issue", "") or "")
        issues_match = stage2_results.get(key, None)  # None if not run

        rows.append({
            "mrn":               mrn,
            "surgeon":           row["surgeon_last"],
            "patient_initials":  row["patient_initials"],
            "feature":           feat,
            "summary_file":      Path(row["summary_path"]).name,
            "gt_is_fabrication": True,
            "pred_is_fabrication": pred_fab,
            "confidence":        confidence,
            "model_issue":       model_issue,
            "gt_issue":          known_issue,
            "issues_match":      issues_match,
        })

df_results = pd.DataFrame(rows)

print("=== VALIDATION RESULTS ===")
print()
display_cols = ["mrn", "surgeon", "patient_initials", "feature",
                "pred_is_fabrication", "confidence", "issues_match"]
print(df_results[display_cols].to_string(index=False))

## 7. Compute Precision / Recall / F1

In [ ]:
# ── Detection metrics (all 14 are GT positives) ───────────────────────────────
valid_mask  = df_results["pred_is_fabrication"].notna()
df_valid    = df_results[valid_mask].copy()

gt_labels   = df_valid["gt_is_fabrication"].astype(bool).tolist()
pred_labels = df_valid["pred_is_fabrication"].astype(bool).tolist()

if len(set(gt_labels)) > 0 and len(pred_labels) > 0:
    precision = precision_score(gt_labels, pred_labels, zero_division=0)
    recall    = recall_score(gt_labels, pred_labels, zero_division=0)
    f1        = f1_score(gt_labels, pred_labels, zero_division=0)
    tp = sum(g and p for g, p in zip(gt_labels, pred_labels))
    fp = sum(not g and p for g, p in zip(gt_labels, pred_labels))
    fn = sum(g and not p for g, p in zip(gt_labels, pred_labels))
    tn = sum(not g and not p for g, p in zip(gt_labels, pred_labels))
else:
    precision = recall = f1 = 0
    tp = fp = fn = tn = 0

# ── Issue match rate (among correctly detected) ───────────────────────────────
detected = df_valid[df_valid["pred_is_fabrication"] == True]
n_issue_evaluated = detected["issues_match"].notna().sum()
n_issue_match     = detected["issues_match"].sum() if n_issue_evaluated > 0 else 0
issue_match_rate  = n_issue_match / n_issue_evaluated if n_issue_evaluated > 0 else None

print("=" * 55)
print(f"  Model              : {VALIDATION_MODEL}")
print(f"  Cases evaluated    : {len(df_valid)} / {len(df_results)}")
print()
print("  FABRICATION DETECTION")
print(f"    TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"    Precision : {precision:.3f}")
print(f"    Recall    : {recall:.3f}")
print(f"    F1 Score  : {f1:.3f}")
print()
print("  ISSUE IDENTIFICATION (among detected positives)")
if issue_match_rate is not None:
    print(f"    Issues match: {int(n_issue_match)}/{int(n_issue_evaluated)} "
          f"= {issue_match_rate:.1%}")
else:
    print("    No issue comparisons run yet")
print("=" * 55)

## 8. Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── A: Detection result bar ───────────────────────────────────────────────────
detect_counts = df_results["pred_is_fabrication"].value_counts(dropna=False)
colors = []
for k in detect_counts.index:
    if k is True:
        colors.append("#e74c3c")
    elif k is False:
        colors.append("#2ecc71")
    else:
        colors.append("#95a5a6")
axes[0].bar(
    [str(k) for k in detect_counts.index],
    detect_counts.values, color=colors, edgecolor="white", linewidth=1.5
)
axes[0].set_title("LLM Fabrication Detection\n(all 14 are GT positives)",
                  fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Predicted is_fabrication")
for i, v in enumerate(detect_counts.values):
    axes[0].text(i, v + 0.1, str(v), ha="center", fontweight="bold")

# ── B: Confidence distribution ────────────────────────────────────────────────
conf_data = df_results.dropna(subset=["confidence"])
if len(conf_data) > 0:
    bar_colors = conf_data["pred_is_fabrication"].map(
        {True: "#e74c3c", False: "#2ecc71", None: "#95a5a6"}
    ).fillna("#95a5a6")
    axes[1].barh(
        range(len(conf_data)),
        conf_data["confidence"].values,
        color=bar_colors,
        edgecolor="white",
    )
    labels = [
        f"{row['surgeon']}/{row['patient_initials']} — {row['feature'][:18]}"
        for _, row in conf_data.iterrows()
    ]
    axes[1].set_yticks(range(len(conf_data)))
    axes[1].set_yticklabels(labels, fontsize=7)
    axes[1].axvline(0.5, color="black", linestyle="--", alpha=0.4, label="0.5 threshold")
    axes[1].set_xlim(0, 1)
    axes[1].set_title("Detection Confidence per Case", fontweight="bold")
    axes[1].set_xlabel("Confidence")
    axes[1].legend(fontsize=8)

# ── C: Metrics bar ────────────────────────────────────────────────────────────
metric_names = ["Precision", "Recall", "F1"]
metric_vals  = [precision, recall, f1]
bar_c = ["#3498db", "#e67e22", "#9b59b6"]
bars = axes[2].bar(metric_names, metric_vals, color=bar_c,
                   edgecolor="white", linewidth=1.5)
axes[2].set_ylim(0, 1.1)
axes[2].set_title(f"Detection Metrics\n({VALIDATION_MODEL})", fontweight="bold")
axes[2].set_ylabel("Score")
for bar, val in zip(bars, metric_vals):
    axes[2].text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                 f"{val:.2f}", ha="center", fontweight="bold")

plt.suptitle(
    f"LLM Fabrication Validation — v1 Summaries  "
    f"(n={len(df_results)} feature-level, model={VALIDATION_MODEL})",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
save_path = REPORTS_DIR / "llm_validation_results.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

## 9. Detailed Results Table

In [ ]:
# Full audit table with model issues vs ground truth
audit_cols = [
    "mrn", "surgeon", "patient_initials", "feature", "summary_file",
    "pred_is_fabrication", "confidence", "issues_match",
    "model_issue", "gt_issue",
]
audit_cols = [c for c in audit_cols if c in df_results.columns]
df_audit   = df_results[audit_cols].sort_values(
    ["pred_is_fabrication", "confidence"], ascending=[False, False]
)

print("=== FULL AUDIT TABLE ===")
print()
for _, row in df_audit.iterrows():
    fab_icon  = "✓ DETECTED" if row["pred_is_fabrication"] else "✗ MISSED"
    conf_str  = f"{row['confidence']:.2f}" if pd.notna(row["confidence"]) else "N/A"
    match_str = str(row["issues_match"]) if pd.notna(row["issues_match"]) else "—"
    print(f"  [{fab_icon}] conf={conf_str} | issue_match={match_str}")
    print(f"    MRN {int(row['mrn'])} {row['surgeon']}/{row['patient_initials']} — {row['feature']}")
    print(f"    Model : {row['model_issue'][:120]}")
    print(f"    GT    : {row['gt_issue'][:120]}")
    print()

audit_path = RUN_OUT_DIR / "audit_table.csv"
df_audit.to_csv(audit_path, index=False)
print(f"Saved: {audit_path}")

## 10. Save Summary Report

In [ ]:
from datetime import datetime

summary = {
    "run_timestamp":     datetime.utcnow().isoformat(),
    "validation_model":  VALIDATION_MODEL,
    "issue_judge_model": ISSUE_JUDGE_MODEL,
    "n_cases":           len(df_results),
    "n_evaluated":       len(df_valid),
    "detection": {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision":  round(precision, 4),
        "recall":     round(recall, 4),
        "f1":         round(f1, 4),
    },
    "issue_identification": {
        "evaluated":   int(n_issue_evaluated),
        "matched":     int(n_issue_match),
        "match_rate":  round(issue_match_rate, 4) if issue_match_rate is not None else None,
    },
}

summary_path = RUN_OUT_DIR / "run_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# Also save full results CSV
results_csv = REPORTS_DIR / "llm_validation_results.csv"
df_results.to_csv(results_csv, index=False)

print("=" * 55)
print("FINAL SUMMARY")
print("=" * 55)
print(json.dumps(summary, indent=2))
print(f"\nOutputs:")
print(f"  {summary_path}")
print(f"  {audit_path}")
print(f"  {results_csv}")
print(f"  {REPORTS_DIR / 'llm_validation_results.png'}")